# Week 7 Day 1 – Machine Learning Dataset Preparation

## Objective

Prepare stock data for supervised machine learning.

Topics covered

- Technical Indicators
- Feature Engineering
- Target Variable
- TimeSeriesSplit

Dataset:
Reliance Industries Historical Stock Data


In [4]:
# Import required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
import yfinance as yf

In [5]:
# Download Reliance historical stock data
reliance = yf.download(
    "RELIANCE.NS",
    start="2020-01-01",
    end="2025-01-01",
    auto_adjust=True
)

# Display first five rows
reliance.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2020-01-01,672.216187,680.008852,670.390532,675.956671,14004468
2020-01-02,683.660278,686.176208,673.284921,673.284921,17710316
2020-01-03,684.484070,686.487896,678.183128,682.636062,20984698
2020-01-06,668.609314,680.365044,667.050769,676.847260,24519177
2020-01-07,678.895569,683.303975,673.952766,676.401873,16683622


## Dataset Information

The cleaned Reliance stock dataset is loaded.

The Date column is converted into DatetimeIndex.

In [6]:
close = reliance["Close"]

close.head()

Ticker,RELIANCE.NS
Date,
2020-01-01,672.216187
2020-01-02,683.660278
2020-01-03,684.484070
2020-01-06,668.609314
2020-01-07,678.895569


In [7]:
# Create SMA indicators

reliance["SMA_10"] = close.rolling(10).mean()

reliance["SMA_20"] = close.rolling(20).mean()

## SMA

Simple Moving Average smooths price movement.

We calculate:

- 10-Day SMA
- 20-Day SMA

In [8]:
# Function to calculate RSI

def RSI(data, period=14):

    delta = data.diff()

    gain = delta.where(delta > 0, 0)

    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(period).mean()

    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    return rsi

In [9]:
reliance["RSI"] = RSI(close)

reliance[["Close","RSI"]].tail()

Price,Close,RSI
Ticker,RELIANCE.NS,
Date,,
2024-12-24,1212.280762,26.460984
2024-12-26,1206.133911,20.062443
2024-12-27,1210.595459,23.413687
2024-12-30,1200.333984,24.276612
2024-12-31,1205.043213,28.121063


## RSI

Relative Strength Index measures momentum.

Range:

- Above 70 → Overbought
- Below 30 → Oversold

In [10]:
ema12 = close.ewm(span=12, adjust=False).mean()

ema26 = close.ewm(span=26, adjust=False).mean()

reliance["MACD"] = ema12 - ema26

reliance["Signal"] = reliance["MACD"].ewm(span=9, adjust=False).mean()

reliance[["MACD","Signal"]].tail()

Price,MACD,Signal
Ticker,,
Date,,
2024-12-24,-22.103150,-16.780639
2024-12-26,-22.943337,-18.013179
2024-12-27,-22.984234,-19.007390
2024-12-30,-23.572926,-19.920497
2024-12-31,-23.389850,-20.614368


## MACD

MACD compares two exponential moving averages.

Features created

- MACD
- Signal Line

In [11]:
# Daily return

reliance["Return"] = close.pct_change()

# Lag features

reliance["Lag_1"] = reliance["Return"].shift(1)

reliance["Lag_5"] = reliance["Return"].shift(5)

reliance["Lag_21"] = reliance["Return"].shift(21)

## Lag Features

Lag returns allow the model to learn from previous market behaviour.

In [12]:
# Predict next day's return

reliance["Target"] = reliance["Return"].shift(-1)

## Target

The target is the next day's stock return.

This makes it a supervised learning problem.

In [13]:
reliance = reliance.dropna()

reliance.head()

Price,Close,High,Low,Open,Volume,SMA_10,SMA_20,RSI,MACD,Signal,Return,Lag_1,Lag_5,Lag_21,Target
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,,,,,,,,,,
Date,,,,,,,,,,,,,,,
2020-01-31,628.599609,647.123856,626.618077,647.012506,34750310,666.229224,674.894650,25.145877,-7.557652,-1.616914,-0.022234,-0.024394,-0.003471,0.017024,-0.018524
2020-02-03,616.955139,623.389665,606.958302,610.008569,30712290,659.690070,672.311942,24.072083,-10.665002,-3.426531,-0.018524,-0.022234,-0.009859,0.001205,0.029520
2020-02-04,635.167725,637.884001,622.432316,623.411935,25383901,654.903156,670.125549,34.425487,-11.525141,-5.046253,0.029520,-0.018524,-0.023099,-0.023192,0.015493
2020-02-05,645.008667,646.923437,636.904365,638.952660,18472100,651.124835,668.686130,36.070456,-11.282667,-6.293536,0.015493,0.029520,0.005504,0.015385,0.006420
2020-02-06,649.149902,653.246611,641.268241,647.502361,15538577,648.050073,666.677850,28.544059,-10.633761,-7.161581,0.006420,0.015493,-0.024394,-0.007510,-0.016566


In [14]:
features = [

    "SMA_10",

    "SMA_20",

    "RSI",

    "MACD",

    "Signal",

    "Lag_1",

    "Lag_5",

    "Lag_21"

]

X = reliance[features]

y = reliance["Target"]

print(X.shape)

print(y.shape)

(1215, 8)
(1215,)


## Feature Matrix

X contains engineered features.

y contains the prediction target.

In [15]:
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_index, test_index) in enumerate(tscv.split(X), start=1):

    print(f"Fold {fold}")

    print("Train:", len(train_index))

    print("Test :", len(test_index))

    print("-"*40)

Fold 1
Train: 205
Test : 202
----------------------------------------
Fold 2
Train: 407
Test : 202
----------------------------------------
Fold 3
Train: 609
Test : 202
----------------------------------------
Fold 4
Train: 811
Test : 202
----------------------------------------
Fold 5
Train: 1013
Test : 202
----------------------------------------


## TimeSeriesSplit

Unlike random train-test split, TimeSeriesSplit preserves the chronological order of observations.

This prevents future data from leaking into the training set and is the recommended approach for financial time-series modeling.

In [16]:
reliance.to_csv("ml_dataset.csv")

print("Dataset saved successfully.")

Dataset saved successfully.


# Conclusion

In this notebook we:

- Loaded cleaned Reliance stock data
- Created technical indicators (SMA, RSI, MACD)
- Generated lag return features
- Created the target variable (next-day return)
- Prepared the feature matrix
- Applied TimeSeriesSplit for model validation
- Saved the processed dataset for future machine learning tasks

The dataset is now ready for training regression and classification models in the next lessons.